# Fairness Over Time - Phase 3: Diagnose

In [3]:
%load_ext autoreload
%autoreload 2

import time
import warnings
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

from src.data import DATA_DIR, ARTIFACTS_DIR, RESULTS_DIR, REPORTS_DIR, METRICS_PATH, CI_PATH, LABEL_DRIFT_PATH, DRIFT_PATH, SHAP_PATH, BACKGROUND_PATH, load_acs, make_race_eth, split_data, save_split, load_split
from src.preprocess import NUMERIC_COLS, ORDINAL_COLS, NOMINAL_COLS, build_preprocessor
from src.models import make_lr, make_xgb, make_pipeline_for, fit_mitigated, load_models, MODEL_INFO
from src.fairness import (make_sensitive, group_metrics, fairness_summary, fairness_long,
                          bootstrap_gaps, evaluate_all, bootstrap_all, save_results)

warnings.filterwarnings("ignore", module="fairlearn")

import shap

In [4]:
models = load_models()

xgb_clf = models["xgb"].named_steps['model']

explainer = shap.TreeExplainer(xgb_clf)

In [5]:
shap.__version__

'0.51.0'

In [6]:
#load one year
X, y, meta =load_acs(2017)

print(((meta["income"] > 50_000).astype(int) == y.squeeze("columns")).all())
print(meta["ADJINC"].unique()) 

# ADJINC puts all incomes in a survey year into that year’s dollars
# CPI-U is used to compare dollars across years.

True
[1011189]


In [7]:
from src.drift import label_rates

X, y, meta = load_acs(2014)
label_rates(meta["income"], 2014)

{'year': 2014,
 'fixed_rate': np.float64(0.3163516715339384),
 'indexed_rate': np.float64(0.3163516715339384)}

In [8]:
YEARS = [2014, 2015, 2016, 2017]
metas = {year: load_acs(year)[2] for year in YEARS}

rates = pd.DataFrame([label_rates(metas[y]["income"], y) for y in YEARS])

base_rate = rates.loc[rates["year"] == 2014, "fixed_rate"].item()
rates["inflation_part"] = rates["fixed_rate"] - rates["indexed_rate"]
rates["real_growth"] = rates['indexed_rate'] - base_rate

rates

,year,fixed_rate,indexed_rate,inflation_part,real_growth
0,2014,0.316352,0.316352,0.000000,0.000000
1,2015,0.331646,0.331328,0.000318,0.014976
2,2016,0.341508,0.339777,0.001732,0.023425
3,2017,0.353854,0.347729,0.006125,0.031378


In [9]:
from src.drift import label_rates_by_group

label_drift = pd.concat(
    [label_rates_by_group(metas[y], y) for y in YEARS], ignore_index=True
)
save_results(label_drift, LABEL_DRIFT_PATH)

# Readable check table (not saved): one row per attribute/group/year
check = label_drift.pivot(index=["attribute", "group", "year"],
                          columns="metric", values="value")
check["inflation_part"] = check["fixed_rate"] - check["indexed_rate"]
check["real_part"] = check["indexed_rate"] - check.groupby(level=["attribute", "group"])["fixed_rate"].transform("first")
check.round(4)

metric                   fixed_rate  indexed_rate  inflation_part  real_part
attribute group    year                                                     
overall   all      2014      0.3164        0.3164          0.0000     0.0000
                   2015      0.3316        0.3313          0.0003     0.0150
                   2016      0.3415        0.3398          0.0017     0.0234
                   2017      0.3539        0.3477          0.0061     0.0314
race_eth  Hispanic 2014      0.1692        0.1692          0.0000     0.0000
                   2015      0.1829        0.1828          0.0001     0.0136
                   2016      0.1887        0.1875          0.0012     0.0184
                   2017      0.2002        0.1957          0.0044     0.0266
          NH Asian 2014      0.4287        0.4287          0.0000     0.0000
                   2015      0.4326        0.4326          0.0000     0.0039
                   2016      0.4470        0.4462          0.0008     0.0175
                   2017      0.4732        0.4706          0.0027     0.0419
          NH Black 2014      0.2190        0.2190          0.0000     0.0000
                   2015      0.2333        0.2332          0.0001     0.0143
                   2016      0.2483        0.2466          0.0017     0.0276
                   2017      0.2671        0.2615          0.0056     0.0425
          NH Other 2014      0.2793        0.2793          0.0000     0.0000
                   2015      0.3100        0.3087          0.0013     0.0294
                   2016      0.3256        0.3247          0.0008     0.0454
                   2017      0.3173        0.3132          0.0041     0.0339
          NH White 2014      0.4109        0.4109          0.0000     0.0000
                   2015      0.4295        0.4290          0.0005     0.0182
                   2016      0.4416        0.4394          0.0022     0.0285
                   2017      0.4525        0.4449          0.0076     0.0340
sex       Female   2014      0.2279        0.2279          0.0000     0.0000
                   2015      0.2427        0.2423          0.0004     0.0145
                   2016      0.2591        0.2573          0.0018     0.0294
                   2017      0.2720        0.2658          0.0062     0.0380
          Male     2014      0.3935        0.3935          0.0000     0.0000
                   2015      0.4090        0.4087          0.0003     0.0153
                   2016      0.4135        0.4118          0.0016     0.0184
                   2017      0.4253        0.4192          0.0061     0.0257

In [10]:
train_idx, val_idx, test_idx = load_split()

Xs = {y: load_acs(y)[0] for y in YEARS}       # [0] = X; slow, run once

X_base = Xs[2014].loc[train_idx]               # what the models learned from

# What each year is compared with X_base. The 2014 entry is the noise floor.
comparisons = {
    2014: Xs[2014].loc[test_idx],              # same year, no real change
    2015: Xs[2015],
    2016: Xs[2016],
    2017: Xs[2017],
}

- PSI (Population Stability Index) compares how a feature’s values are spread across bins in two datasets:

    PSI = Σ over bins of (cur% − base%) × ln(cur% / base%)

- It’s 0 if the shares are identical, and it grows when shares move, especially when a bin that was rare becomes common (or the reverse).

In [11]:
from src.drift import make_bins, assign_bins, psi

b = make_bins(X_base["AGEP"], numeric=True)
base_b = assign_bins(X_base["AGEP"], b, numeric=True)

print("self    :", psi(base_b, base_b))                                                   # must be 0.0
print("noise   :", psi(base_b, assign_bins(comparisons[2014]["AGEP"], b, numeric=True)))  # 2014 test
print("2017    :", psi(base_b, assign_bins(comparisons[2017]["AGEP"], b, numeric=True)))
print(base_b.value_counts(normalize=True).sort_index())                                   # ~10% per bin?

self    : 0.0
noise   : 0.00024580676119014244
2017    : 0.0015750969071915218
AGEP
(-inf, 23.0]    0.113430
(23.0, 28.0]    0.099353
(28.0, 33.0]    0.103879
(33.0, 38.0]    0.101374
(38.0, 43.0]    0.100980
(43.0, 47.0]    0.082810
(47.0, 52.0]    0.108038
(52.0, 57.0]    0.108511
(57.0, 62.0]    0.087651
(62.0, inf]     0.093974
Name: proportion, dtype: float64


In [12]:
from src.drift import feature_drift

drift = pd.concat(
    [feature_drift(X_base, X_cur, year) for year, X_cur in comparisons.items()],
    ignore_index=True,
)
print(len(drift))
save_results(drift, DRIFT_PATH)

# Readable view: one row per feature/metric, one column per year
view = drift.pivot(index=["metric", "feature"], columns="year", values="value")
view["x_noise_2017"] = view[2017] / view[2014]   # 2017 drift as a multiple of the noise floor
view.sort_values(["metric", 2017], ascending=[True, False]).round(4)

52


year               2014    2015    2016    2017  x_noise_2017
metric feature                                               
KS     SCHL      0.0028  0.0055  0.0170  0.0279        9.9943
       WKHP      0.0027  0.0071  0.0097  0.0097        3.5314
       AGEP      0.0041  0.0042  0.0068  0.0091        2.2236
PSI    OCCP      0.0161  0.0101  0.0122  0.0167        1.0373
       SCHL      0.0009  0.0011  0.0021  0.0052        6.0046
       POBP      0.0046  0.0027  0.0034  0.0048        1.0446
       race_eth  0.0000  0.0005  0.0010  0.0023     4069.4706
       AGEP      0.0002  0.0003  0.0007  0.0016        6.4079
       WKHP      0.0002  0.0005  0.0009  0.0014        9.0871
       RELP      0.0010  0.0008  0.0011  0.0011        1.0593
       COW       0.0008  0.0005  0.0006  0.0007        0.8426
       MAR       0.0001  0.0002  0.0007  0.0006        7.3367
       SEX       0.0000  0.0000  0.0000  0.0000       28.9787

## SHAP

In [14]:
N_SHAP = 1_000
N_BACKGROUND = 500
SEED = 42

# 1,000 rows per year to explain. comparisons[2014] is the 2014 TEST set, as in Phase 2.
shap_X = {year: X_cur.sample(N_SHAP, random_state=SEED) for year, X_cur in comparisons.items()}

# Background: 500 raw rows from 2014 TRAIN. Used by LR's explainer now and by the app later.
background = X_base.sample(N_BACKGROUND, random_state=SEED)
background.to_csv(BACKGROUND_PATH, index=False)

print({y: len(x) for y, x in shap_X.items()})

{2014: 1000, 2015: 1000, 2016: 1000, 2017: 1000}


In [22]:
#run shap on xgboost in 2014
from src.explain import explain
shap_14 = explain(models["xgb"], shap_X[2014], background)

print(shap_14.shape)                                         # expect (1000, 10)
shap_14.abs().mean().sort_values(ascending=False)

(1000, 10)


AGEP        0.636628
SCHL        0.618378
WKHP        0.588411
OCCP        0.547921
SEX         0.299521
RELP        0.224680
race_eth    0.210652
MAR         0.136367
COW         0.123498
POBP        0.048417
dtype: float32

In [24]:
#run shap on linear regression in 2014
shap_lr14 = explain(models["lr"], shap_X[2014], background)

# Additivity: per person, sum of SHAP = log-odds − average background log-odds
pre, lr = models["lr"].named_steps["pre"], models["lr"].named_steps["model"]
expected = lr.decision_function(pre.transform(shap_X[2014])) \
           - lr.decision_function(pre.transform(background)).mean()
print(np.allclose(shap_lr14.sum(axis=1), expected, atol=1e-6))   # expect True

shap_lr14.abs().mean().sort_values(ascending=False)

True


OCCP        0.876476
SCHL        0.452687
AGEP        0.443841
WKHP        0.424264
RELP        0.385836
SEX         0.336718
MAR         0.198544
race_eth    0.152090
COW         0.141146
POBP        0.086724
dtype: float64

In [26]:
from src.explain import shap_importance

SHAP_MODELS = {"lr": "LR", "xgb": "XGBoost"}      # unmitigated only; labels match MODEL_INFO

shap_values = {}                                   # keep per-person SHAP for the optional group view
rows = []
for year, X_s in shap_X.items():
    for name, label in SHAP_MODELS.items():
        sv = explain(models[name], X_s, background)
        shap_values[(year, label)] = sv
        rows.append(shap_importance(sv, year, label))

shap_imp = pd.concat(rows, ignore_index=True)
print(len(shap_imp))                               # expect 4 × 2 × 10 × 2 = 160
save_results(shap_imp, SHAP_PATH)

view = shap_imp[shap_imp["metric"] == "mean_abs_shap"].pivot(
    index=["model", "feature"], columns="year", values="value")
view["change_14_17"] = view[2017] - view[2014]
view.sort_values(["model", 2014], ascending=[True, False]).round(3)

160


year               2014   2015   2016   2017  change_14_17
model   feature                                           
LR      OCCP      0.876  0.871  0.877  0.852        -0.025
        SCHL      0.453  0.473  0.481  0.482         0.030
        AGEP      0.444  0.452  0.451  0.436        -0.008
        WKHP      0.424  0.403  0.411  0.437         0.012
        RELP      0.386  0.385  0.387  0.376        -0.010
        SEX       0.337  0.336  0.337  0.337        -0.000
        MAR       0.199  0.203  0.205  0.200         0.001
        race_eth  0.152  0.151  0.150  0.150        -0.002
        COW       0.141  0.131  0.135  0.137        -0.004
        POBP      0.087  0.087  0.089  0.088         0.001
XGBoost AGEP      0.637  0.620  0.616  0.588        -0.049
        SCHL      0.618  0.639  0.634  0.640         0.022
        WKHP      0.588  0.539  0.555  0.589         0.000
        OCCP      0.548  0.542  0.545  0.534        -0.014
        SEX       0.300  0.300  0.299  0.302         0.002
        RELP      0.225  0.218  0.216  0.223        -0.002
        race_eth  0.211  0.212  0.211  0.207        -0.003
        MAR       0.136  0.141  0.139  0.139         0.003
        COW       0.123  0.116  0.120  0.121        -0.002
        POBP      0.048  0.050  0.050  0.050         0.002

In [27]:
shap_values[(2017, "XGBoost")].abs().std() / np.sqrt(1000)

AGEP        0.019482
WKHP        0.018556
SCHL        0.010603
COW         0.003880
MAR         0.003192
OCCP        0.014130
POBP        0.002332
RELP        0.006856
SEX         0.002800
race_eth    0.002719
dtype: float32